In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ENVIRONMENT BOOTSTRAP
# Detects Google Colab, clones the repository, wires sys.path and downloads
# the dataset from a public Google Drive folder via gdown.
# In a local environment this cell is a no-op.
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, subprocess, shutil

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

REPO_NAME       = 'global-ads-performance'
GITHUB_URL      = f'https://github.com/pmusachio/{REPO_NAME}.git'
DRIVE_FOLDER_ID = '1gHuzMGXVx34A6unXbc1kq9bKrcCOgmXu'
DATA_FALLBACK   = (
    f'https://raw.githubusercontent.com/pmusachio/{REPO_NAME}/main/data/raw.csv'
)

if _in_colab():
    REPO_PATH = f'/content/{REPO_NAME}'

    if not os.path.exists(REPO_PATH):
        print('Cloning repository...')
        subprocess.run(
            ['git', 'clone', '--quiet', GITHUB_URL, REPO_PATH],
            check=True, capture_output=True,
        )

    os.chdir(REPO_PATH)
    if REPO_PATH not in sys.path:
        sys.path.insert(0, REPO_PATH)

    os.makedirs('data',   exist_ok=True)
    os.makedirs('assets', exist_ok=True)

    if not os.path.exists('data/raw.csv'):
        print('Downloading dataset from Google Drive...')
        subprocess.run(
            ['pip', 'install', '-q', 'gdown'],
            check=True, capture_output=True,
        )
        import gdown

        _tmp = '/content/_gdrive_tmp'
        try:
            gdown.download_folder(
                f'https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}',
                output=_tmp,
                quiet=True,
                remaining_ok=True,
            )
            _src = os.path.join(_tmp, REPO_NAME)
            for _f in os.listdir(_src):
                shutil.move(os.path.join(_src, _f), os.path.join('data', _f))
            shutil.rmtree(_tmp, ignore_errors=True)
        except Exception:
            import urllib.request
            urllib.request.urlretrieve(DATA_FALLBACK, 'data/raw.csv')
            print('Dataset downloaded from GitHub fallback URL.')

    print(f'Working directory : {os.getcwd()}')
    print(f'sys.path[0]       : {sys.path[0]}')
    status = 'OK' if os.path.exists('data/raw.csv') else 'ERROR — file not found'
    print(f'Dataset           : {status}')
else:
    print('Local environment detected — no bootstrap required.')

# Global Ads Performance — Statistical Analysis and Budget Optimization

**Objective:** Determine whether campaign type (Search vs. Video) produces a statistically significant
difference in ROAS for e-commerce advertisers, then fit per-channel revenue response models and
solve the constrained budget allocation problem.

| Step | Technique |
|------|-----------|
| A/B test design | Sample-size power analysis (Cohen's formula) |
| Assumption validation | Shapiro-Wilk / D'Agostino-Pearson · Levene's test |
| Inferential test | Welch's t-test or Mann-Whitney U (data-driven) |
| Channel modeling | Adstock decay + Saturation regression (L, k) |
| Budget optimization | Constrained SLSQP via SciPy |


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy  as np
import pandas as pd
import matplotlib.pyplot    as plt
import matplotlib.gridspec  as gridspec
from scipy        import stats
from scipy.stats  import shapiro, levene, mannwhitneyu, ttest_ind, norm
from sklearn.metrics  import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline

from src.transformers import AdstockTransformer, SaturationRegressor, saturation_curve
from src.optimizer    import optimize_allocation

# Global plot style (Catppuccin Mocha palette)
plt.rcParams.update({
    'figure.facecolor': '#1e1e2e',
    'axes.facecolor':   '#1e1e2e',
    'axes.edgecolor':   '#45475a',
    'axes.labelcolor':  '#cdd6f4',
    'xtick.color':      '#cdd6f4',
    'ytick.color':      '#cdd6f4',
    'text.color':       '#cdd6f4',
    'grid.color':       '#313244',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'legend.facecolor': '#1e1e2e',
    'legend.edgecolor': '#45475a',
})
PALETTE = {
    'Google Ads':  '#89dceb',
    'Meta Ads':    '#a6e3a1',
    'TikTok Ads':  '#f38ba8',
    'Search':      '#89b4fa',
    'Video':       '#fab387',
    'Shopping':    '#cba6f7',
}
SEED = 42
np.random.seed(SEED)

print('Libraries loaded.')

## 1. Data Loading and Exploratory Analysis

The dataset records daily advertising performance per platform, campaign type and industry.
Key columns: `platform`, `campaign_type`, `industry`, `ad_spend`, `revenue`, `ROAS`, `conversions`, `CTR`.


In [ ]:
df = pd.read_csv('data/raw.csv', parse_dates=['date'])

print(f'Shape          : {df.shape}')
print(f'Date range     : {df["date"].min().date()} -> {df["date"].max().date()}')
print(f'Industries     : {sorted(df["industry"].unique())}')
print(f'Platforms      : {sorted(df["platform"].unique())}')
print(f'Campaign types : {sorted(df["campaign_type"].unique())}')
print(f'Missing values :\n{df.isnull().sum()[df.isnull().sum() > 0]}')
print()
df.describe().round(3)

In [ ]:
fig = plt.figure(figsize=(16, 9))
fig.suptitle('Advertising Performance Overview — All Industries', fontsize=14, color='#cdd6f4', y=1.01)
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.50, wspace=0.38)

# 1. ROAS distribution by platform
ax1 = fig.add_subplot(gs[0, 0])
for plat in sorted(df['platform'].unique()):
    ax1.hist(
        df[df['platform'] == plat]['ROAS'].dropna(),
        bins=30, alpha=0.65, label=plat, color=PALETTE.get(plat, '#cdd6f4'),
    )
ax1.set_title('ROAS Distribution by Platform', fontsize=10)
ax1.set_xlabel('ROAS')
ax1.set_ylabel('Frequency')
ax1.legend(fontsize=8)
ax1.grid(True)

# 2. Median ROAS by campaign type
ax2 = fig.add_subplot(gs[0, 1])
ct_roas = df.groupby('campaign_type')['ROAS'].median().sort_values(ascending=False)
bars    = ax2.bar(
    ct_roas.index, ct_roas.values,
    color=[PALETTE.get(c, '#b4befe') for c in ct_roas.index],
    edgecolor='#45475a',
)
ax2.set_title('Median ROAS by Campaign Type', fontsize=10)
ax2.set_ylabel('Median ROAS')
ax2.bar_label(bars, fmt='%.2f', color='#cdd6f4', fontsize=9)
ax2.grid(True, axis='y')

# 3. Ad Spend vs Revenue scatter
ax3 = fig.add_subplot(gs[0, 2])
for plat in sorted(df['platform'].unique()):
    sub = df[df['platform'] == plat]
    ax3.scatter(
        sub['ad_spend'], sub['revenue'],
        alpha=0.3, s=15, label=plat, color=PALETTE.get(plat, '#cdd6f4'),
    )
ax3.set_title('Ad Spend vs Revenue', fontsize=10)
ax3.set_xlabel('Ad Spend ($)')
ax3.set_ylabel('Revenue ($)')
ax3.legend(fontsize=8)
ax3.grid(True)

# 4. Daily total ad spend timeline
ax4 = fig.add_subplot(gs[1, :2])
daily_total = df.groupby('date')['ad_spend'].sum()
ax4.fill_between(daily_total.index, daily_total.values, alpha=0.35, color='#89b4fa')
ax4.plot(daily_total.index, daily_total.values, color='#89b4fa', linewidth=1.2)
ax4.set_title('Total Daily Ad Spend Over Time', fontsize=10)
ax4.set_xlabel('Date')
ax4.set_ylabel('Ad Spend ($)')
ax4.grid(True)

# 5. Pearson correlation heatmap
ax5   = fig.add_subplot(gs[1, 2])
cols  = ['ad_spend', 'conversions', 'CTR', 'ROAS', 'revenue']
corr  = df[cols].corr()
im    = ax5.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
ax5.set_xticks(range(len(cols)))
ax5.set_yticks(range(len(cols)))
ax5.set_xticklabels(cols, rotation=45, ha='right', fontsize=8)
ax5.set_yticklabels(cols, fontsize=8)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax5.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center',
                 fontsize=7, color='#1e1e2e')
ax5.set_title('Correlation Matrix', fontsize=10)
plt.colorbar(im, ax=ax5, fraction=0.046, pad=0.04)

plt.savefig('assets/01_eda_overview.png', dpi=150, bbox_inches='tight', facecolor='#1e1e2e')
plt.show()
print('Saved -> assets/01_eda_overview.png')

## 2. A/B Test Design

**Business question:** Do Search campaigns achieve a statistically different ROAS compared to Video
campaigns in the e-commerce vertical?

| Parameter | Value |
|-----------|-------|
| Metric | ROAS (Return On Ad Spend) |
| Group A | Search campaigns |
| Group B | Video campaigns |
| H0 | μ_Search = μ_Video |
| H1 | μ_Search ≠ μ_Video (two-sided) |
| Significance level (α) | 0.05 |
| Statistical power (1−β) | 0.80 |

The significance threshold of 5% means we accept a 5% probability of a false positive.
A power of 80% means the test has an 80% chance of detecting a real effect of the observed magnitude.


In [ ]:
df_ecom = df[df['industry'] == 'E-commerce'].copy()
group_a = df_ecom[df_ecom['campaign_type'] == 'Search']['ROAS'].dropna()
group_b = df_ecom[df_ecom['campaign_type'] == 'Video']['ROAS'].dropna()

desc = pd.DataFrame({
    'Group A (Search)': group_a.describe(),
    'Group B (Video)':  group_b.describe(),
}).round(4)
print('Descriptive statistics:')
print(desc.to_string())
print()

# A priori sample size — Cohen's formula for two-sample t-test
pooled_std = np.sqrt((group_a.std()**2 + group_b.std()**2) / 2)
cohen_d    = abs(group_a.mean() - group_b.mean()) / pooled_std

ALPHA, POWER = 0.05, 0.80
z_alpha = norm.ppf(1 - ALPHA / 2)
z_beta  = norm.ppf(POWER)
n_req   = int(np.ceil(2 * ((z_alpha + z_beta) / cohen_d) ** 2))

magnitude = 'small' if cohen_d < 0.5 else 'medium' if cohen_d < 0.8 else 'large'
print(f'Effect size (Cohen\'s d)   : {cohen_d:.4f}  [{magnitude}]')
print(f'Required n per group      : {n_req}')
print(f'Observed n — Group A      : {len(group_a)}  [{"SUFFICIENT" if len(group_a) >= n_req else "INSUFFICIENT"}]')
print(f'Observed n — Group B      : {len(group_b)}  [{"SUFFICIENT" if len(group_b) >= n_req else "INSUFFICIENT"}]')

## 3. Assumption Validation

Before choosing the test statistic, we must validate:

1. **Normality** — Shapiro-Wilk (n ≤ 5 000) or D'Agostino-Pearson (n > 5 000).  
   If both groups are normal → parametric Welch's t-test.  
   If either group is non-normal → non-parametric Mann-Whitney U.

2. **Variance homogeneity** (only if both groups are normal) — Levene's test.  
   Equal variance → Student's t-test. Unequal → Welch's t-test.


In [ ]:
def normality_test(data, label):
    if len(data) <= 5_000:
        stat, p = shapiro(data)
        name = 'Shapiro-Wilk'
    else:
        stat, p = stats.normaltest(data)
        name = "D'Agostino-Pearson"
    result = 'Normal' if p > 0.05 else 'Non-normal'
    print(f'  {name:<22} {label:<20} stat={stat:.4f}  p={p:.4f}  [{result}]')
    return p > 0.05

print(f'Normality tests (alpha = {ALPHA}):')
normal_a    = normality_test(group_a, 'Group A (Search)')
normal_b    = normality_test(group_b, 'Group B (Video)')
both_normal = normal_a and normal_b
print()

equal_var = False
if both_normal:
    lev_stat, lev_p = levene(group_a, group_b)
    equal_var = lev_p > 0.05
    hom = 'Homogeneous' if equal_var else 'Heterogeneous'
    print(f'Levene\'s test (variances)   stat={lev_stat:.4f}  p={lev_p:.4f}  [{hom}]')
else:
    print('Levene\'s test — skipped (at least one group is non-normal).')
print()

# Visualise distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Assumption Validation: ROAS Distributions (E-commerce)', fontsize=13, color='#cdd6f4')

for ax, data, label, color in zip(
    axes[:2],
    [group_a, group_b],
    ['Group A — Search', 'Group B — Video'],
    [PALETTE['Search'], PALETTE['Video']],
):
    ax.hist(data, bins=30, color=color, alpha=0.75, edgecolor='#45475a')
    mu, sigma = data.mean(), data.std()
    x_line = np.linspace(data.min(), data.max(), 300)
    bin_w   = (data.max() - data.min()) / 30
    ax.plot(
        x_line, stats.norm.pdf(x_line, mu, sigma) * len(data) * bin_w,
        color='#f5e0dc', linewidth=2, linestyle='--', label='Normal fit',
    )
    ax.axvline(mu, color='#f38ba8', linewidth=1.5, linestyle=':', label=f'Mean={mu:.2f}')
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('ROAS')
    ax.legend(fontsize=8)
    ax.grid(True)

axes[2].boxplot(
    [group_a.values, group_b.values],
    labels=['Search', 'Video'],
    patch_artist=True,
    boxprops=dict(facecolor='#313244', color='#cdd6f4'),
    medianprops=dict(color='#a6e3a1', linewidth=2),
    whiskerprops=dict(color='#cdd6f4'),
    capprops=dict(color='#cdd6f4'),
    flierprops=dict(marker='o', color='#f38ba8', alpha=0.5, markersize=4),
)
axes[2].set_title('ROAS Side-by-Side', fontsize=10)
axes[2].set_ylabel('ROAS')
axes[2].grid(True, axis='y')

plt.tight_layout()
plt.savefig('assets/02_assumption_validation.png', dpi=150, bbox_inches='tight', facecolor='#1e1e2e')
plt.show()
print('Saved -> assets/02_assumption_validation.png')

## 4. Inferential Statistical Test

The test is selected automatically based on the assumption-validation outcomes above:

| Condition | Test chosen |
|-----------|-------------|
| Both normal, equal variance | Student's t-test |
| Both normal, unequal variance | Welch's t-test |
| At least one non-normal | Mann-Whitney U |


In [ ]:
if both_normal:
    test_label = "Welch's t-test" if not equal_var else "Student's t-test"
    t_stat, p_value = ttest_ind(group_a, group_b, equal_var=equal_var)
    stat_label, stat_val = 't-statistic', t_stat
else:
    test_label = 'Mann-Whitney U'
    u_stat, p_value = mannwhitneyu(group_a, group_b, alternative='two-sided')
    stat_label, stat_val = 'U-statistic', u_stat

# Effect size
if both_normal:
    effect_val, effect_lbl = cohen_d, "Cohen's d"
else:
    # Rank-biserial correlation
    effect_val = 1 - 2 * stat_val / (len(group_a) * len(group_b))
    effect_lbl = 'Rank-biserial r'

# 95% confidence interval for the difference in means
diff_mean = group_a.mean() - group_b.mean()
se_diff   = np.sqrt(group_a.var(ddof=1)/len(group_a) + group_b.var(ddof=1)/len(group_b))
ci_lo, ci_hi = diff_mean - 1.96 * se_diff, diff_mean + 1.96 * se_diff

print('=' * 56)
print('  STATISTICAL TEST RESULTS')
print('=' * 56)
print(f'  Test applied      : {test_label}')
print(f'  {stat_label:<18}: {stat_val:.4f}')
print(f'  p-value           : {p_value:.4f}')
print(f'  {effect_lbl:<18}: {effect_val:.4f}')
print(f'  Mean difference   : {diff_mean:+.4f}')
print(f'  95% CI            : [{ci_lo:.4f}, {ci_hi:.4f}]')
print('=' * 56)

if p_value < ALPHA:
    direction = 'higher' if diff_mean > 0 else 'lower'
    print(f'\n  DECISION: Reject H0 (p < {ALPHA})')
    print(f'  Search campaigns yield a statistically significant')
    print(f'  {direction} ROAS than Video campaigns in e-commerce.')
else:
    print(f'\n  DECISION: Fail to reject H0 (p >= {ALPHA})')
    print(f'  No statistically significant difference detected.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('A/B Test — ROAS: Search vs Video Campaigns (E-commerce)', fontsize=13, color='#cdd6f4')

# KDE comparison
for data, label, color in [
    (group_a, 'Group A — Search', PALETTE['Search']),
    (group_b, 'Group B — Video',  PALETTE['Video']),
]:
    kde = stats.gaussian_kde(data)
    x   = np.linspace(data.min() - 1, data.max() + 1, 300)
    axes[0].fill_between(x, kde(x), alpha=0.25, color=color)
    axes[0].plot(x, kde(x), color=color, linewidth=2, label=f'{label} (n={len(data)})')
    axes[0].axvline(data.mean(), color=color, linewidth=1.5, linestyle=':')
axes[0].set_title(f'KDE — p-value = {p_value:.4f}', fontsize=10)
axes[0].set_xlabel('ROAS')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=9)
axes[0].grid(True)

# Confidence interval for the difference
x_range = np.linspace(ci_lo - abs(ci_lo)*1.5, ci_hi + abs(ci_hi)*1.5, 400)
dist     = stats.norm(loc=diff_mean, scale=se_diff)
axes[1].plot(x_range, dist.pdf(x_range), color='#cba6f7', linewidth=2)
axes[1].fill_between(
    x_range,
    dist.pdf(x_range),
    where=(x_range >= ci_lo) & (x_range <= ci_hi),
    alpha=0.35, color='#89b4fa', label='95% CI',
)
axes[1].axvline(diff_mean, color='#a6e3a1', linewidth=2, linestyle='--',
                label=f'Diff = {diff_mean:+.3f}')
axes[1].axvline(0, color='#f38ba8', linewidth=1.5, linestyle=':', label='H0 (diff = 0)')
axes[1].set_title('Sampling Distribution of the Mean Difference', fontsize=10)
axes[1].set_xlabel('Difference in ROAS (A - B)')
axes[1].set_ylabel('Probability Density')
axes[1].legend(fontsize=9)
axes[1].grid(True)

plt.tight_layout()
plt.savefig('assets/03_ab_test_result.png', dpi=150, bbox_inches='tight', facecolor='#1e1e2e')
plt.show()
print('Saved -> assets/03_ab_test_result.png')

## 5. Channel Revenue Response Modeling

We fit a two-stage per-channel pipeline to the aggregated daily spend/revenue data:

1. **AdstockTransformer** — geometric carryover decay:  
   `effective_spend[t] = spend[t] + decay_rate * effective_spend[t-1]`

2. **SaturationRegressor** — diminishing-returns saturation curve:  
   `revenue = L * (1 - exp(-k * spend))`  
   where `L` is the asymptotic revenue ceiling and `k` is the saturation speed.

A chronological 70/30 split is used for training and held-out evaluation,
preserving temporal order to avoid look-ahead bias.


In [ ]:
channels = sorted(df_ecom['platform'].unique())
daily = (
    df_ecom
    .groupby(['date', 'platform'])
    .agg(ad_spend=('ad_spend', 'sum'), revenue=('revenue', 'sum'))
    .reset_index()
)

fitted = {}   # channel -> {pipeline, L, k, r2, rmse, X, y, train_end}
header = f'{"Channel":<14} {"L (ceiling $)":>15} {"k (saturation)":>15} {"R2 test":>9} {"RMSE test":>13}'
print(header)
print('-' * len(header))

for ch in channels:
    cdf = daily[daily['platform'] == ch].sort_values('date').reset_index(drop=True)
    X   = cdf['ad_spend'].values.reshape(-1, 1)
    y   = cdf['revenue'].values

    n         = len(cdf)
    train_end = int(n * 0.70)

    pipeline = Pipeline([
        ('adstock', AdstockTransformer(decay_rate=0.3)),
        ('model',   SaturationRegressor(L_init=float(y.max()) * 1.5, k_init=5e-4)),
    ])
    pipeline.fit(X[:train_end], y[:train_end])

    y_pred = np.maximum(pipeline.predict(X[train_end:]), 0)
    r2     = r2_score(y[train_end:], y_pred)
    rmse   = np.sqrt(mean_squared_error(y[train_end:], y_pred))

    model = pipeline.named_steps['model']
    fitted[ch] = dict(pipeline=pipeline, L=model.L_, k=model.k_,
                      r2=r2, rmse=rmse, X=X, y=y, train_end=train_end)

    print(f'{ch:<14} {model.L_:>15,.2f} {model.k_:>15.6f} {r2:>9.4f} {rmse:>13,.2f}')

In [ ]:
fig, axes = plt.subplots(1, len(channels), figsize=(5 * len(channels), 5), sharey=False)
fig.suptitle('Saturation Curves — Revenue Response to Ad Spend per Channel',
             fontsize=13, color='#cdd6f4')

if len(channels) == 1:
    axes = [axes]

for ax, ch in zip(axes, channels):
    res   = fitted[ch]
    color = PALETTE.get(ch, '#b4befe')

    x_range = np.linspace(0, float(res['X'].max()) * 1.1, 300)
    ax.plot(x_range, saturation_curve(x_range, res['L'], res['k']),
            color=color, linewidth=2.5, label='Saturation curve')

    te = res['train_end']
    ax.scatter(res['X'][:te], res['y'][:te], alpha=0.35, s=18,
               color=color, label='Train')
    ax.scatter(res['X'][te:], res['y'][te:], alpha=0.90, s=28,
               color='#f5e0dc', edgecolors=color, linewidths=0.8, label='Test')

    ax.axhline(res['L'], color='#a6e3a1', linestyle='--', linewidth=1,
               label=f'Ceiling L={res["L"]:,.0f}')
    ax.set_title(f'{ch}\nR2={res["r2"]:.3f}  |  k={res["k"]:.5f}', fontsize=10)
    ax.set_xlabel('Daily Ad Spend ($)')
    ax.set_ylabel('Daily Revenue ($)')
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.savefig('assets/04_saturation_curves.png', dpi=150, bbox_inches='tight', facecolor='#1e1e2e')
plt.show()
print('Saved -> assets/04_saturation_curves.png')

## 6. Budget Optimization

Given the fitted saturation curves, we solve a constrained optimization problem:

**Maximize** total predicted revenue across channels  
**Subject to:**
- Total allocation ≤ daily budget cap  
- Global ROAS ≥ minimum acceptable threshold  
- Each channel receives between `min_share` and `max_share` of the total budget  

The solver uses SLSQP (Sequential Least Squares Programming) from `scipy.optimize.minimize`.


In [ ]:
TOTAL_BUDGET = 15_000
MIN_ROAS     = 2.0
MIN_SHARE    = 0.10
MAX_SHARE    = 0.60

channel_params = {ch: {'L': res['L'], 'k': res['k']} for ch, res in fitted.items()}

opt = optimize_allocation(
    channel_params=channel_params,
    total_budget=TOTAL_BUDGET,
    min_roas=MIN_ROAS,
    min_share=MIN_SHARE,
    max_share=MAX_SHARE,
)

if not opt['success']:
    print('Optimization failed to converge — constraints may be too strict.')
else:
    print(f'Optimal allocation for a ${TOTAL_BUDGET:,} daily budget (min ROAS {MIN_ROAS}x):\n')
    for ch, spend in opt['allocation'].items():
        pct = spend / opt['estimated_spend'] * 100
        print(f'  {ch:<14}  ${spend:>8,.2f}  ({pct:.1f}%)')
    print()
    print(f'  Estimated Revenue : ${opt["estimated_revenue"]:>10,.2f}')
    print(f'  Estimated Spend   : ${opt["estimated_spend"]:>10,.2f}')
    print(f'  Estimated ROAS    : {opt["estimated_roas"]:>6.2f}x')

In [ ]:
if opt['success']:
    ch_list = list(opt['allocation'].keys())
    alloc   = [opt['allocation'][c] for c in ch_list]
    colors  = [PALETTE.get(c, '#b4befe') for c in ch_list]

    fig, (ax_bar, ax_sat) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        f'Optimal Budget Allocation — ${TOTAL_BUDGET:,} daily budget | min ROAS {MIN_ROAS}x',
        fontsize=13, color='#cdd6f4',
    )

    # Allocation bar chart
    bars = ax_bar.barh(ch_list, alloc, color=colors, edgecolor='#45475a')
    ax_bar.set_xlabel('Allocated Spend ($)')
    ax_bar.set_title('Recommended Allocation', fontsize=11)
    for bar, val in zip(bars, alloc):
        ax_bar.text(
            bar.get_width() + TOTAL_BUDGET * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f'${val:,.0f}', va='center', fontsize=10, color='#cdd6f4',
        )
    ax_bar.set_xlim(0, max(alloc) * 1.35)
    ax_bar.grid(True, axis='x')

    # Operating points on saturation curves
    x_range = np.linspace(0, TOTAL_BUDGET, 300)
    for ch, spend, color in zip(ch_list, alloc, colors):
        p        = channel_params[ch]
        y_curve  = saturation_curve(x_range, p['L'], p['k'])
        y_at_opt = saturation_curve(spend, p['L'], p['k'])
        ax_sat.plot(x_range, y_curve, color=color, linewidth=2, alpha=0.75, label=ch)
        ax_sat.scatter([spend], [y_at_opt], color=color, s=110,
                       edgecolors='#cdd6f4', zorder=5)
        ax_sat.vlines(spend, 0, y_at_opt, color=color, linestyle=':', alpha=0.5)

    ax_sat.set_xlabel('Daily Ad Spend ($)')
    ax_sat.set_ylabel('Predicted Daily Revenue ($)')
    ax_sat.set_title('Operating Points on Saturation Curves', fontsize=11)
    ax_sat.legend(fontsize=9)
    ax_sat.grid(True)

    plt.tight_layout()
    plt.savefig('assets/05_optimization_result.png', dpi=150, bbox_inches='tight', facecolor='#1e1e2e')
    plt.show()
    print('Saved -> assets/05_optimization_result.png')

## 7. Business Recommendation

### A/B Test Conclusion

The inferential test compared ROAS between Search and Video campaign types for e-commerce advertisers.
The result (reported above) indicates whether the observed performance gap is statistically reliable
or attributable to random sampling variation.

- If **H0 was rejected**: budget planning should treat the campaign types as genuinely different
  and prioritise the type with the higher ROAS, while monitoring for confounders (seasonality,
  audience overlap, budget parity).
- If **H0 was not rejected**: the data does not support claiming one format is superior;
  allocate budget based on strategic goals (brand awareness vs. direct conversion) rather than
  ROAS alone.

### Optimization Conclusion

The SLSQP optimizer found the spend allocation that maximises predicted revenue given the
saturation curves fitted to historical data, subject to a total budget cap, a minimum ROAS
floor and per-channel share bounds.

The saturation parameters `(L, k)` carry concrete business meaning:

| Parameter | Interpretation |
|-----------|----------------|
| `L` | Asymptotic revenue ceiling — the maximum revenue the channel can yield regardless of spend |
| `k` | Saturation speed — higher `k` means the channel reaches diminishing returns at lower spend levels |

Channels with a high `L` but low `k` are the most scalable and should receive a larger share
of any budget increase. Channels already close to saturation (operating near `L`) should
receive marginal or no additional investment.
